# 🧔‍♂️🎮👩 Gender Representation in Video Games 👩🎮🧔‍♂️ 

## Context:
These are top-selling or top-rated games, so the numbers describe successful games, not the whole industry.

## Objective: Analizing the representation of women in Videogames. This analyze will be run through the lense of 3 main questions :

| **Q1** | How much more are women sexualized than men, in what ways, and does it depend on the role they play ? 
| **Q2** | What roles do women hold: lead, playable, romantic interest? 
| **Q3** | Good vs. bad: are women allies, antagonists or main villains? 

### Angles that can be added to each question

Each of Q1–Q3 can be re-read through these three lenses:

| **A. Time** | Has it changed over time (2012–2022)? 
| **B. Team mix** | Does the team's gender mix change what is shown?
| **C. Country** | Does the developer's country have an influence? 

# 📥 Data Loading

### 🎯 Goal
Understand what each table contains (size, columns, types, typical values) before cleaning anything.

### 📥 Input
Three CSV files downloaded with `kagglehub` from the Kaggle dataset `br33sa/gender-representation-in-video-games`:

| Table | One row is... | Links to |
|---|---|---|
| `games` | a game | `characters.Game` |
| `characters` | a relevant character | `games.Game_Id`, `sexualization.Id` |
| `sexualization` | the sexualization rating of a character | `characters.Id` |

### 🛠️ What we do
1. Import the libraries (`pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `kagglehub`, `os`) and set the plot defaults: `sns.set_style("whitegrid")` and figure size `(10, 6)`.
2. Download the dataset with `kagglehub.dataset_download("br33sa/gender-representation-in-video-games")` and store the folder in `path`.
3. Read each CSV with `pd.read_csv(os.path.join(path, ...))` into `games`, `characters` and `sexualization`.
4. For each table, check `shape`, `columns.tolist()`, `head()` and `info()`.
5. Summarize with `describe()` (numeric columns) and `describe(exclude='number')` (text columns).

### 📝 Observations
*(Fill in after running the code.)*
- e.g. `Unnamed: 27` is completely empty: a column to drop.
- e.g. `Percentage_non_male` and `Team_percentage` are stored as text.
- Full column definitions: `docs/data-dictionary.md`.

## 1. Imports and plot defaults

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Download the dataset

In [ ]:
path = kagglehub.dataset_download("br33sa/gender-representation-in-video-games")
print("path to files:", path)

## 3. Read each CSV

In [ ]:
games = pd.read_csv(os.path.join(path, "games.grivg.csv"))
characters = pd.read_csv(os.path.join(path, "characters.grivg.csv"))
sexualization = pd.read_csv(os.path.join(path, "sexualization.grivg.csv"))

tables = {"games": games, "characters": characters, "sexualization": sexualization}

descriptions = {
    "games": "Information about the games. Each row represents a game. Includes genre, development team, and critic reviews.",
    "characters": "Information about the characters. Each row represents a relevant character. Joins with `games` by `Game`.",
    "sexualization": "Information about the sexualization ratings of characters. Each row represents a rating for a character. Joins by `Id`.",
}

## 4. Check structure of each table

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}\n\n{descriptions[name]}"))
    print(f"shape: {d.shape}\n")
    print(f"columns: {d.columns.tolist()}\n")
    display(d.head())
    d.info()
    print("\n")

## 5. Descriptive statistics

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}"))
    display(d.describe())                       # numeric columns
    display(d.describe(exclude="number"))       # text columns

# 🔍 Exploración

> We look for problems in the data (missing values, duplicates, outliers, inconsistencies).

---

### 🎯 Goal
Find what needs cleaning before changing anything. Studying the variables comes later, in *Análisis*, once the tables are clean.

### 📥 Input
The three tables loaded in *Carga* (`games`, `characters`, `sexualization`), still unmodified.

### 🛠️ What we do
1. **Missing values:** count nulls per column (`df.isnull().sum()`) and show the percentage (`nulls / len(df) * 100`), only for columns above 0. Optional: `missingno` matrix.
2. **Duplicates:** `df.duplicated().sum()`.
3. **Outliers:** IQR rule, flagged outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`, plus boxplots.
4. **Dubious data:** check consistency between columns (e.g. `characters.Sexualization` vs `sexualization.Total`) and impossible values (e.g. ages).

### 📊 Charts checklist
Every chart has a title, named axes, and a legend when it has more than one series. Minimum **5 visualizations** in total.

### 📝 Observations and decisions
*(Fill in after running the code. Nothing is changed in this section: decisions are applied in *Limpieza*.)*

| Finding | Where | What we will do | Why |
|---|---|---|---|
| *(e.g. `Series` has 58% nulls)* | `games` | *(keep / fill / drop)* | *(reason)* |
| *(e.g. outlier in `Total_team`)* | `games` | *(keep / cap / drop)* | *(reason)* |

## 1. Missing values

In [ ]:
import missingno as msno

for name, d in tables.items():
    display(Markdown(f"#### {name}"))

    nulls = d.isnull().sum()
    nulls = nulls[nulls > 0]

    if nulls.empty:
        print("No missing values.\n")
        continue

    report = pd.DataFrame({
        "missing": nulls,
        "percent": (nulls / len(d) * 100).round(1),
    }).sort_values("percent", ascending=False)
    display(report)

    ax = msno.matrix(d, sparkline=False)
    ax.set_title(f"Missing values in {name}")
    plt.show()



## 2. Duplicates

In [ ]:
for name, d in tables.items():
    display(Markdown(f"#### {name}"))
    n_duplicates = d.duplicated().sum()

    if n_duplicates == 0:
        print("No duplicates found.\n")
        continue

    print(f"{n_duplicates} duplicated row(s):")
    display(d[d.duplicated(keep=False)]) 
    tables[name] = d.drop_duplicates()

## 3. Outliers

> 🚧 **Draft.** First pass on `Team_percentage` (boxplot in *Análisis*). To be completed (IQR on every numeric variable, final decisions).

### Team_percentage (share of women in the development team)

**Method:** IQR rule on the boxplot. A value is an outlier if it lies outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.

**Findings**<br>
- Q1 = 0%, median = 0%, Q3 = 14.75%, so IQR = 14.75 and the upper fence is about 36.9%.
- 5 of the 64 games lie above it: 43%, 50%, 60% and two at 67%.
- The distribution is strongly right-skewed (skewness ≈ 1.9): 36 of the 64 games (56%) have exactly 0% women in the team, and the mean (10.7%) is far above the median (0%).

⚖️ **Decision (provisional):** keep the 5 outlier games.<br>
**Why:** they are not data errors. They are real games with unusually balanced teams, and they are the most interesting cases for the team-mix question. Removing them would also leave only 59 games in a dataset that is already small.

⚠️ **Caution:** because the box is squashed at 0, the IQR rule flags a lot relative to the typical value. The mean is misleading here, so we use the median.

### Avg_Reviews
*(to fill in: outliers seen on the boxplot, and decision)*

### To revisit
- Check the other numeric variables (`Total_team`, `Protagonist`, `Percentage_non_male`, `Age`...).
- Confirm the decision on `Team_percentage` once we run angle B (see Q1–Q3 add-ons).

## 4. Dubious data

### Columns Metacritic, Destructoid, IGN, GameSpot and Avg_Reviews
Column "Avg_Reviews" which is supposed to be the average of the 4 review scores ("Metacritic ", "Destructoid", "IGN", "GameSpot")
We could consider dropping the 4 review score columns if Avg_Reviews is reliable.

In [ ]:
# First checking how Avg_Reviews was computed.

scores = ["Metacritic ", "Destructoid", "IGN", "GameSpot"]   # note the trailing space in "Metacritic "
recomputed = games[scores].mean(axis=1).round(1)
print("Games where Avg_Reviews differs from the recomputed mean:",
      (recomputed != games["Avg_Reviews"]).sum())


In [ ]:
# checking the distribution of the differences

diff = (recomputed - games["Avg_Reviews"]).round(2)
mismatch = games.loc[diff != 0, ["Title"] + scores + ["Avg_Reviews"]].copy()
mismatch["recomputed"] = recomputed[diff != 0]
mismatch["diff"] = diff[diff != 0]
display(mismatch)

**Findings**<br>

8 games differ by 0.1, and in all 8 the exact mean ends in 5 (e.g. 9.45), so the source rounds up and Python rounds down. 

It is only a rounding difference, so `Avg_Reviews` is reliable. The other columns can be dropped in the cleaning section

### Sexualization table
Column "Total" from Sexialization table is supposed to be the average of the 4 other columns ("Sexualized_clothing", "Trophy", "Damsel in Distress", "Sexualized Cutscenes")
We should make sure "Total" is reliable before using it in the analyze.

In [ ]:
sex = tables["sexualization"]
chars = tables["characters"]
criteria = ["Sexualized_clothing", "Trophy", "Damsel in Distress", "Sexualized Cutscenes"]

#Every criterion must be 0 or 1
print("Values outside {0,1}:", (~sex[criteria].isin([0, 1])).sum().sum())

#Total must equal the sum of the four criteria
print("Total != sum of criteria:", (sex["Total"] != sex[criteria].sum(axis=1)).sum())

#characters.Sexualization must equal sexualization.Total
merged = chars.merge(sex, on="Id", how="outer", indicator=True)
print("Ids missing in one table:", (merged["_merge"] != "both").sum())
print("Sexualization != Total:", (merged["Sexualization"] != merged["Total"]).sum())

#Metadata rule: a child or infant is never a "damsel in distress"
display(merged.query("Age_range in ['Child', 'Infant'] and `Damsel in Distress` == 1")
              [["Name", "Game", "Age_range", "Damsel in Distress", "Total"]])

games_c = tables["games"]

**Findings**<br> 

Checks 1 to 3 all give 0, so the tables are fully consistent. 

Check 4 finds 1 real inconsistency: the character Son (game PP, a Child) has Damsel_in_distress = 1, which contradicts the author's own rule. (Children are expected to require assistance and are therefore not considered as "Damsel in distress") 

It's 1 of 637 rows, so it won't change the results, but it should be cleaned.

### Age vs Age_range
The numeric ages must fall inside their category

In [ ]:
age = pd.to_numeric(chars["Age"], errors="coerce")
labels = ["Infant", "Child", "Teenager", "Young adult", "Adult", "Middle-aged", "Elderly"]
expected = pd.cut(age, bins=[0, 6, 15, 18, 25, 40, 65, np.inf], right=False, labels=labels).astype(str)
print("Age outside its Age_range:", (age.notnull() & (expected != chars["Age_range"])).sum())

### Team_percentage 
Team_percentage must match Female_team / Total_team

In [ ]:
pct = games_c["Team_percentage"].str.rstrip("%").astype(float)
print("Team_percentage off by more than 1 point:",
      ((pct - games_c["female_team"] / games_c["Total_team"] * 100).abs() > 1).sum())

# 🧹 Data Cleaning

> We turn the raw tables into clean, analysis-ready ones: fixing types, filling or dropping gaps, removing columns we don't need and giving columns consistent names.

---

### 🎯 Goal
Apply the decisions taken in *Exploración*, so the analysis starts from tidy tables. Every change is explained (what we did and why) and the raw tables stay untouched.

### 📥 Input
The three tables from *Carga* (`games`, `characters`, `sexualization`) and the problems found in *Exploración* (empty column, gaps in `Series`, text values in numeric columns, inconsistent names).

### 🛠️ What we do
1. **Remove empty or useless columns:** `Unnamed: 27` (empty) and the columns no question uses (`Sub-genre`, `Publisher`, `Platform`, `PEGI`, `Criteria`), and `Age` from `characters` (duplicated by `Age_range`).
2. **Keep one review score:** keep `Avg_Reviews` and drop the four individual scores (`Metacritic`, `Destructoid`, `IGN`, `GameSpot`), once we verified it is reliable.
3. **Fill gaps with meaning:** `Series` → `"Standalone"` (a missing value means "no series").
4. **Rename columns** to one consistent style (`Female_team`, `Damsel_in_distress`, `Sexualized_cutscenes`).
5. **Fix data types:** `Release` text (`Nov-13`) → date and year; `Percentage_non_male` and `Team_percentage` (`"11%"`) → numbers; `Age` (numbers and words) → numeric where possible.
6. **Create new variables** (at least 3), for example `Release_year`, a female-lead flag, and groups of countries.
7. **Check the result:** `info()` on the clean tables, and no missing value left unexplained.

### 🔎 Rules we follow
| Situation | Treatment |
|---|---|
| A blank means "does not apply" (e.g. no series) | Fill with a label |
| A numeric value is truly unknown | Leave as `NaN`, never a text label |
| A column is empty or not used by any question | Drop it, with the reason |
| Two columns hold the same information | Keep one, after checking they match |

### 📝 Decisions
Each decision is written next to the code that applies it, as **what we did + why**.

### ✅ Result
*(Fill in after running the code.)*
- Clean tables: `games_clean`, `characters_clean`, `sexualization_clean`.
- Columns before → after, per table.

Notes:
- Steps 5 and 6 haven't been done in your notebook yet (the types and the new variables are tasks #3 and #14). Remove them from the list for now if you'd rather document only what is finished, or keep them as the plan and fill them in later.
- The "Rules we follow" table is only a summary of the principles you already applied. Delete it if you prefer the section shorter.
- Your current notebook does part of this work (the drops, the Series fill) inside Exploración. If you want the structure to match, move those cells under this Limpieza heading. Then Exploración only finds problems, and Limpieza only fixes them.

## 1. Remove empty or useless columns:

### Unnamed 

⚖️ **Decision:** drop `Unnamed: 27` from `games`. <br>
    **Why:** the column is completely empty. It comes from a trailing comma at the end of every row in the CSV, so it holds no information.

In [ ]:
tables["games"] = tables["games"].drop(columns=["Unnamed: 27"], errors="ignore")
print(tables["games"].shape)

### Useless variables

⚖️ **Decision:** drop `Sub-genre`, `Publisher`, `Platform`, `PEGI` and `Criteria`.<br>
    **Why:** none of our three questions or their angles (time, team mix, country) uses them.

In [ ]:
cols_to_drop = ["Sub-genre", "Publisher", "Platform", "PEGI", "Criteria"]
tables["games"] = tables["games"].drop(columns=cols_to_drop)
print(tables["games"].columns.tolist())

### Age from Characters table

⚖️ **Decision:** drop `Age` from `characters`. `Age_range` is the column we keep.<br>

**Why:** `Age` mixes exact numbers (150 characters) with words (487 characters). The words are identical to `Age_range`, and every numeric age falls inside its `Age_range` as defined in the metadata. So `Age_range` already holds the same information, complete for all 637 characters and with no missing values. None of our questions needs the exact age.

In [ ]:
characters = tables["characters"]

# Evidence before dropping: the words in Age equal Age_range, and Age_range has no gaps
age_num = pd.to_numeric(characters["Age"], errors="coerce")
words = characters.loc[age_num.isnull()]
print("Word values in Age that differ from Age_range:", (words["Age"] != words["Age_range"]).sum())
print("Missing Age_range:", characters["Age_range"].isnull().sum())

tables["characters"] = characters.drop(columns=["Age"])
print(tables["characters"].columns.tolist())


## 2. Keep one review score

⚖️ **Decision:** drop `Metacritic`, `Destructoid`, `IGN`, and `GameSpot`. Keep `Avg_Reviews`<br>
    **Why:** `Avg_Reviews` is the average of the four scores and has no gaps, while the individual scores have missing values. 
    
    `Avg_Reviews` has been found reliable in previous investigation.

In [ ]:
cols_to_drop = ["Metacritic ", "Destructoid", "IGN", "GameSpot"]   # note the trailing space in "Metacritic "
tables["games"] = tables["games"].drop(columns=cols_to_drop)
print(tables["games"].columns.tolist())

## 3. Fill gaps with meaning

⚖️ **Decision: fill the missing values of `Series` with `"Standalone"`.**<br>

    **Why:** a missing value here does not mean "unknown", it means the game belongs to no series. A label keeps these 37 games (58%) in `groupby` and charts, which ignore NaN by default.

In [ ]:
# Series: a missing value means the game belongs to no series, so we label it
games = tables["games"].copy()
games["Series"] = games["Series"].fillna("Standalone")
tables["games"] = games

# Check: no missing values left in those columns
print("Missing in Series:", games["Series"].isnull().sum())
print(games["Series"].value_counts().head(), "\n")

## 4. Rename columns

⚖️ **Decision:** Renaming columns whose name has a space, a hyphen, or does not start with a capital letter .<br>

    **Why:** Having a consistent naming convention. No other renaming seems necessary.

In [ ]:
# Finding columns whose name has a space, a hyphen, or does not start with a capital letter

for name, d in tables.items():
    bad = [c for c in d.columns if c != c.strip() or " " in c or "-" in c or not c[0].isupper()]
    print(f"{name}: {bad if bad else 'names OK'}")
    
rename_map = {
    "games": {"female_team": "Female_team"},
    "sexualization": {
        "Damsel in Distress": "Damsel_in_distress",
        "Sexualized Cutscenes": "Sexualized_cutscenes",
    },
}

for name, mapping in rename_map.items():
    tables[name] = tables[name].rename(columns=mapping)

print(tables["sexualization"].columns.tolist())

d.columns.tolist()
d.head()

## 5. Fix Data Type

### Release from Games table

⚖️ **Decision:** convert `Release` from text (`"Nov-13"`) to a date, and create `Release_year`.<br>

**Why:** the time angle (A) of Q1–Q3 needs the year, and text can't be sorted or plotted over time. The months are English abbreviations, so we map them with an explicit English dictionary instead of relying on the computer's language settings.

In [ ]:
games = tables["games"].copy()

month_map = {m: i for i, m in enumerate(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

month = games["Release"].str[:3].map(month_map)
year = 2000 + games["Release"].str[-2:].astype(int)     # "13" -> 2013 (games are 2012-2022)

games["Release"] = pd.to_datetime(pd.DataFrame({"year": year, "month": month, "day": 1}))
games["Release_year"] = games["Release"].dt.year
tables["games"] = games

# Check: no missing date, range is 2012-2022
print("Missing dates:", games["Release"].isnull().sum())
print(games["Release"].min(), "->", games["Release"].max())

### `Percentage_non_male` and `Team_percentage` from Games table

⚖️ **Decision:** convert `Percentage_non_male` and `Team_percentage` from text (`"11%"`) to numbers (0-100).<br>

**Why:** a percentage stored as text can't be averaged, plotted or compared. We keep the 0-100 scale so values read as percentages.

In [ ]:
games = tables["games"].copy()

for col in ["Percentage_non_male", "Team_percentage"]:
    games[col] = games[col].str.rstrip("%").astype(float)

tables["games"] = games
print(games[["Percentage_non_male", "Team_percentage"]].dtypes)

In [ ]:
# Final check of the types
tables["games"].info()

# 📊 Análisis

> With clean tables, we study the variables alone (univariate) and in pairs (bivariate) to answer Q1–Q3.

---

### 🎯 Goal
Describe each variable we need, then compare groups (mainly women vs men) to answer our three questions.

### 📥 Input
The clean tables from *Limpieza* (`games`, `characters`, `sexualization`).

### 🛠️ What we do
1. **Univariate analysis** (at least 3 variables):
   - categorical (e.g. `Gender`, `Relevance`) → bar chart of `value_counts()`
   - numeric (e.g. `Team_percentage`) → histogram + boxplot
2. **Bivariate analysis** (at least 3 relationships):
   - category vs category → `pd.crosstab(..., normalize="index") * 100` + grouped bars (e.g. `Gender` × `Sexualization`)
   - number vs category → boxplot by group
3. Show `n` next to every percentage: the dataset is small.

### 📊 Charts checklist
Every chart has a title, named axes, and a legend when it has more than one series.


## 1. Univariate analysis

### Gender Variable from Characters table

In [ ]:
characters = tables["characters"]

# 1. The numbers
counts = characters["Gender"].value_counts()
summary = pd.DataFrame({
    "n": counts,
    "percent": (counts / len(characters) * 100).round(1),
})
display(summary)

# 2. Bar chart
fig, ax = plt.subplots()
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_title("Characters by gender (637 characters)")
ax.set_xlabel("Gender")
ax.set_ylabel("Number of characters")

# write "n (percent)" above each bar
for i, n in enumerate(counts.values):
    ax.text(i, n + 5, f"{n} ({n / len(characters) * 100:.1f}%)", ha="center")

plt.show()

**Findings**<br>

- men are about 62% of characters (393 of 637) and women about 32% (203). So there are roughly two men for every woman.
- Non-binary and Custom are smaller groups (33 and 8) 

### Team percentage from Games table

In [ ]:
games = tables["games"]

# Convert "11%" -> 11.0
games = tables["games"]
team_pct = games["Team_percentage"]

# 1. The numbers
print(team_pct.describe())
print("Median:", team_pct.median())
print("Skewness:", round(team_pct.skew(), 2))
print("Games with 0%:", (team_pct == 0).sum(), f"({(team_pct == 0).mean() * 100:.0f}%)")

# 2. Histogram
fig, ax = plt.subplots()
sns.histplot(team_pct, bins=15, ax=ax)
ax.axvline(team_pct.mean(), color="red", linestyle="--", label=f"Mean ({team_pct.mean():.1f}%)")
ax.axvline(team_pct.median(), color="green", linestyle="-", label=f"Median ({team_pct.median():.1f}%)")
ax.set_title("Share of women in the development team (64 games)")
ax.set_xlabel("Team_percentage (%)")
ax.set_ylabel("Number of games")
ax.legend()
plt.show()

# 3. Boxplot
fig, ax = plt.subplots(figsize=(10, 3))
sns.boxplot(x=team_pct, ax=ax)
ax.set_title("Boxplot of the share of women in the development team")
ax.set_xlabel("Team_percentage (%)")
plt.show()

**Findings**<br>

- Mean vs. median: the mean (10.7) is well above the median (0). This gap is the sign of a skewed distribution, and a few high values are pulling the mean up.

- Histogram: one tall bar at 0 and a long tail to the right.

- Boxplot: the box is squashed at 0 because 25% and 50% are both 0 and the 75% mark is only 14.75. The dots on the right (43, 50, 60, 67, 67) are flagged as outliers by the IQR rule.

- The typical game in this dataset has no women in its core team, and the mean misleads here, so we will prefer the median.

### Roles from characters

In [ ]:
characters = tables["characters"]

role_labels = {
    "PA": "Protagonist",
    "DA": "Deuteragonist",
    "SK": "Sidekick",
    "MC": "Main character",
    "SC": "Secondary character",
    "MA": "Main antagonist",
}

# 1. The numbers
counts = characters["Relevance"].value_counts()
summary = pd.DataFrame({
    "role": counts.index.map(role_labels),
    "n": counts.values,
    "percent": (counts.values / len(characters) * 100).round(1),
}, index=counts.index)
display(summary)

# 2. Bar chart (sorted from most to least frequent)
fig, ax = plt.subplots()
sns.barplot(x=counts.values, y=counts.index.map(role_labels), ax=ax)
ax.set_title("Characters by role in the story (637 characters)")
ax.set_xlabel("Number of characters")
ax.set_ylabel("Role (Relevance)")

for i, n in enumerate(counts.values):
    ax.text(n + 3, i, f"{n} ({n / len(characters) * 100:.1f}%)", va="center")

plt.show()

**Findings**<br>

- The majority are background: secondary characters are about 42% (266 of 637). Main characters are another 32%.

- Protagonists are rare: 80 characters, about 13%. 


## 2. Bivariate analysis

# OTHER

- **Angle B and `Team_percentage` is heavily skewed.** 36 of the 64 games (56%) have exactly 0% women in the team, and the median is 0%. Correlating or averaging the raw percentage is therefore misleading. Instead, compare two groups: games with **0%** vs games with **>0%** (n = 36 vs 28), and show `n` for both. If the >0% group is still informative, it can be split further (for example, 1–25% and >25%).